# IPSR Unit 3: Problem Solving Techniques and Agents

Subject: Introduction to Problem Solving and Reasoning (B.Tech Semester I).

Nine topics: the four agent types, breadth-first search, and learning agents. All agents here live in the VacBot world so you can compare them fairly.

**How to use this notebook:** read a section, run every code cell with Shift+Enter, then try the exercises. Each exercise cell has only comments; write your answer there. Solutions are hidden: click *Show solution* only after you have tried.

---
## 3.1 Description of an agent

*HTML lab: Unit 3 lab, lesson 1*

An agent is a function from **percept history** to **action**. The *agent program* is the code that implements that function. Agent = architecture (body and sensors) + program. We reuse one small world for every agent type: two squares A and B, each dirty or clean.

In [1]:
import random
def run(agent, steps=20, seed=3, show=0, p_dirty=0.5):
    """Vacuum world. +10 per dirt cleaned, -1 per move, dirt may reappear."""
    rnd = random.Random(seed)
    dirt = {"A": rnd.random() < .7, "B": rnd.random() < .7}
    loc, score, log = "A", 0, []
    for t in range(steps):
        percept = (loc, dirt[loc])
        action = agent(percept)
        if action == "Suck":
            if dirt[loc]: score += 10
            dirt[loc] = False
        elif action == "Right": loc, score = "B", score - 1
        elif action == "Left":  loc, score = "A", score - 1
        for s in "AB":
            if not dirt[s] and rnd.random() < 0.05: dirt[s] = True     # dirt reappears
        log.append((t, percept, action, score))
    for row in log[:show]: print(row)
    return score

**Exercise 1.** Write an agent that always answers 'Suck'. Run it and print the score.

In [2]:
# Exercise 1: write your code below
# Hint: def agent(percept): return ...


<details><summary>Show solution</summary>

```python
always_suck = lambda percept: "Suck"
print(run(always_suck))
```

</details>

---
## 3.2 Reflex agent diagram and program

*HTML lab: Unit 3 lab, lesson 2*

**Simple reflex agent:** percept, then condition-action rules, then action. No memory. Diagram: *sensors -> what the world is like now -> condition-action rules -> actuators*.

In [3]:
def reflex(percept):
    loc, dirty = percept
    if dirty: return "Suck"
    return "Right" if loc == "A" else "Left"
print("score:", run(reflex, show=6))

(0, ('A', True), 'Suck', 10)
(1, ('A', False), 'Right', 9)
(2, ('B', True), 'Suck', 19)
(3, ('B', False), 'Left', 18)
(4, ('A', True), 'Suck', 28)
(5, ('A', False), 'Right', 27)
score: 24


It works, but it keeps moving after everything is clean. It cannot know that, because it only sees its own square.

**Exercise 1.** Add a rule table as a dict `{('A',True):'Suck', ...}` and rebuild the same agent from it.

In [4]:
# Exercise 1: write your code below
# Hint: 4 percept combinations


<details><summary>Show solution</summary>

```python
table = {("A",True):"Suck", ("A",False):"Right", ("B",True):"Suck", ("B",False):"Left"}
table_agent = lambda p: table[p]
print(run(table_agent) == run(reflex))
```

</details>

---
## 3.3 Model-based reflex agent diagram and program

*HTML lab: Unit 3 lab, lesson 3*

**Model-based reflex agent** adds an internal **state**: what it remembers about the world, updated with a model of how the world works. Diagram: *sensors -> state (+ how the world evolves, + what my actions do) -> rules -> actuators*.

In [5]:
def make_model_based():
    known = {"A": None, "B": None}        # internal state: what I believe about each square
    def agent(percept):
        loc, dirty = percept
        known[loc] = dirty
        if dirty:
            known[loc] = False            # model of my action: sucking cleans this square
            return "Suck"
        if known["A"] is False and known["B"] is False:
            return "NoOp"                 # I believe everything is clean: stop moving
        return "Right" if loc == "A" else "Left"
    return agent
print("reflex      :", run(reflex))
print("model-based :", run(make_model_based()))

reflex      : 24
model-based : 19


Model-based wins here because it stops wasting moves. (A `NoOp` counts as no move, no cost.)

**Exercise 1.** Make the model-based agent also say 'NoOp' when only the *other* square is clean and this one has just been cleaned. Does the score change? Explain in a comment.

In [6]:
# Exercise 1: write your code below
# Hint: try it and compare


<details><summary>Show solution</summary>

```python
# Answer: it does not change, because 'known' already marks both squares clean after Suck,
# and the NoOp rule covers that case. The idea is that memory replaces repeated sensing.
print(run(make_model_based()))
```

</details>

---
## 3.4 Goal-based agent diagram

*HTML lab: Unit 3 lab, lesson 4*

**Goal-based agent:** adds a **goal**. It asks "what will happen if I do action A?" and "will that bring me closer to my goal?". Diagram: *state + model -> predict result of each action -> compare with goals -> action*. It chooses by looking ahead, not by fixed rules.

In [7]:
def goal_based(percept, goal_cells=("A", "B")):
    loc, dirty = percept
    def result(action, loc, dirty):          # tiny model of what each action does
        if action == "Suck": return loc, False
        return ("B" if action == "Right" else "A"), None
    # prefer an action that achieves "clean here"
    if dirty and result("Suck", loc, dirty)[1] is False: return "Suck"
    return "Right" if loc == "A" else "Left"
print("goal-based:", run(goal_based))

goal-based: 24


**Exercise 1.** In words (a comment), say one difference between this agent and the reflex agent even though they pick the same actions here.

In [8]:
# Exercise 1: write your code below
# Hint: think: how are actions chosen?


<details><summary>Show solution</summary>

```python
# The goal-based agent chooses by predicting results and checking them against a goal,
# so if the goal changes (e.g. only clean B) we change the goal, not all the rules.
```

</details>

---
## 3.5 Utility-based agent diagram

*HTML lab: Unit 3 lab, lesson 4*

**Utility-based agent:** goals only say success or failure. **Utility** gives a *number* for how good a state is, so the agent can trade off (fast vs safe). Diagram: *predict result of each action -> utility of that state -> choose best expected utility*.

In [9]:
def utility_based(percept):
    loc, dirty = percept
    options = {"Suck": 10 if dirty else -0.5, "Left": -1 if loc == "B" else -3, "Right": -1 if loc == "A" else -3}
    return max(options, key=options.get)
print("utility-based:", run(utility_based))
routes = {"Highway": (30, 0.2), "City": (45, 0.05)}      # (minutes, risk of accident)
for r, (mins, risk) in routes.items():
    print(r, "utility =", round(-mins - 400 * risk, 1))

utility-based: 20
Highway utility = -110.0
City utility = -65.0


**Exercise 1.** Change the accident penalty from 400 to 50. Which route wins now?

In [10]:
# Exercise 1: write your code below
# Hint: re-compute utility


<details><summary>Show solution</summary>

```python
for r, (mins, risk) in routes.items():
    print(r, round(-mins - 50 * risk, 1))
# Highway wins: with a small penalty speed matters more than risk.
```

</details>

---
## 3.6 Breadth-first search

*HTML lab: Unit 3 lab, lesson 5*

BFS explores a graph level by level using a **queue** (first in, first out). It finds the path with the fewest steps. Keep a **frontier** (nodes waiting) and an **explored** set (nodes already seen).

In [11]:
from collections import deque
graph = {
 "S": ["A", "B"], "A": ["C", "D"], "B": ["D", "E"],
 "C": [], "D": ["G"], "E": ["G"], "G": [],
}
def bfs(graph, start, goal):
    frontier = deque([[start]])
    explored = {start}
    while frontier:
        path = frontier.popleft()
        node = path[-1]
        if node == goal: return path
        for nxt in graph[node]:
            if nxt not in explored:
                explored.add(nxt)
                frontier.append(path + [nxt])
    return None
print(bfs(graph, "S", "G"))
print(bfs(graph, "S", "Z") if "Z" in graph else "Z is not in the graph")

['S', 'A', 'D', 'G']
Z is not in the graph


**Exercise 1.** Find the shortest path from A to G.

In [12]:
# Exercise 1: write your code below
# Hint: bfs(graph, 'A', 'G')


<details><summary>Show solution</summary>

```python
print(bfs(graph, "A", "G"))
```

</details>

---
## 3.7 BFS worked example with the frontier queue

*HTML lab: Unit 3 lab, lesson 5*

The same search, but printing the frontier at every step so you can see the queue change.

In [13]:
def bfs_trace(graph, start, goal):
    frontier = deque([start]); explored = {start}; parent = {start: None}
    step = 0
    while frontier:
        step += 1
        node = frontier.popleft()
        print(f"step {step}: take {node:2} | ", end="")
        if node == goal:
            print("GOAL")
            path = []
            while node: path.append(node); node = parent[node]
            return path[::-1]
        for nxt in graph[node]:
            if nxt not in explored:
                explored.add(nxt); parent[nxt] = node; frontier.append(nxt)
        print("frontier =", list(frontier))
print("path:", bfs_trace(graph, "S", "G"))

step 1: take S  | frontier = ['A', 'B']
step 2: take A  | frontier = ['B', 'C', 'D']
step 3: take B  | frontier = ['C', 'D', 'E']
step 4: take C  | frontier = ['D', 'E']
step 5: take D  | frontier = ['E', 'G']
step 6: take E  | frontier = ['G']
step 7: take G  | GOAL
path: ['S', 'A', 'D', 'G']


**Exercise 1.** Add a node H connected from E (E -> H) and trace BFS to H.

In [14]:
# Exercise 1: write your code below
# Hint: graph['E'].append('H') and graph['H'] = []


<details><summary>Show solution</summary>

```python
graph["E"].append("H"); graph["H"] = []
print(bfs_trace(graph, "S", "H"))
```

</details>

---
## 3.8 BFS time and space complexity

*HTML lab: Unit 3 lab, lesson 5*

With branching factor **b** and goal depth **d**, BFS may create about 1 + b + b^2 + ... + b^d nodes, so time and space are both **O(b^d)**. Space is the real problem because every frontier node is stored. BFS is **complete** (finds a solution if one exists) and **optimal** when every step costs the same.

In [15]:
b = 10
print(f"{'depth':>5} {'nodes':>18} {'memory at 1 KB/node':>22}")
for d in range(0, 9, 2):
    nodes = sum(b**i for i in range(d + 1))
    mb = nodes / 1024
    mem = f"{mb:,.0f} MB" if mb < 1024 else f"{mb/1024:,.1f} GB"
    print(f"{d:>5} {nodes:>18,} {mem:>22}")

depth              nodes    memory at 1 KB/node
    0                  1                   0 MB
    2                111                   0 MB
    4             11,111                  11 MB
    6          1,111,111                 1.1 GB
    8        111,111,111               106.0 GB


**Exercise 1.** How many nodes at depth 5 when b = 3?

In [16]:
# Exercise 1: write your code below
# Hint: sum of powers


<details><summary>Show solution</summary>

```python
print(sum(3**i for i in range(6)))
```

</details>

---
## 3.9 Learning agents and learning agent diagram

*HTML lab: Unit 3 lab, lesson 6*

A **learning agent** has four parts: the **performance element** (chooses actions), the **critic** (scores how well it did against a standard), the **learning element** (improves the performance element using the critic's feedback) and the **problem generator** (suggests new things to try, so it keeps exploring). Below, a tiny agent learns which square gets dirty more often.

In [17]:
import random
class LearningVac:
    def __init__(self, seed=0):
        self.rnd = random.Random(seed)
        self.value = {"A": 0.0, "B": 0.0}      # learned: how worthwhile it is to wait at each square
        self.counts = {"A": 0, "B": 0}
    def choose(self, epsilon=0.2):             # performance element + problem generator
        if self.rnd.random() < epsilon: return self.rnd.choice("AB")      # explore
        return max(self.value, key=self.value.get)                       # exploit
    def learn(self, square, reward):           # learning element, fed by the critic's reward
        self.counts[square] += 1
        self.value[square] += (reward - self.value[square]) / self.counts[square]

true_dirt_rate = {"A": 0.2, "B": 0.7}          # unknown to the agent
env = random.Random(42)
agent = LearningVac()
for t in range(300):
    sq = agent.choose()
    reward = 10 if env.random() < true_dirt_rate[sq] else 0     # critic: dirt found?
    agent.learn(sq, reward)
print("learned values:", {k: round(v, 2) for k, v in agent.value.items()})
print("visits:", agent.counts)
print("agent now prefers square", max(agent.value, key=agent.value.get))

learned values: {'A': 2.5, 'B': 6.96}
visits: {'A': 60, 'B': 240}
agent now prefers square B


**Exercise 1.** Set epsilon to 0 (never explore) and run again. What can go wrong?

In [18]:
# Exercise 1: write your code below
# Hint: change choose(epsilon=0)


<details><summary>Show solution</summary>

```python
agent0 = LearningVac(); env = random.Random(42)
for t in range(300):
    sq = agent0.choose(epsilon=0); reward = 10 if env.random() < true_dirt_rate[sq] else 0
    agent0.learn(sq, reward)
print(agent0.counts)
# With no exploring it can lock onto the first square it tried and never discover that B is better.
```

</details>

**Where to read more:** AIMA chapter 2 (agent types) and chapter 3 (search); Visualgo or any BFS animation for practice